In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import col

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/streaming/user_data/users.csv")
)

In [0]:
df.limit(5).display()

id,transaction_date,purchase_amount,product_category,store_id
1,2025-01-01,120.5,Electronics,S001
2,2025-01-02,45.0,Books,S002
3,2025-01-03,89.99,Clothing,S001
4,2025-01-04,230.1,Home,S003
5,2025-01-05,15.75,Books,S002


#### There is a csv file and a API endpoint(REST API)
- The API is used to enrich the csv file 
- To get the records in the API, the id column of the csv file is used
- To get this, first convert the id column into a list
- To convert the column into list, use .collect method that outputs a list of row elements eg: [Row(column_name = value), .....]
- This row element behaves same way as a dictionary i.e "column_name": "value"
- So the values can be accessed using row_element["column_name"]

##### Extracting the id column data into a list

In [0]:
user_ids = [
    row["id"]
    for row in df.select("id").collect()
]

#### Creating function for data ingestion from the api

In [0]:
import requests
import json

base_url = "https://dummyjson.com/users/"

def api_hit():
    api_response = []
    for user in user_ids:
        response = requests.get(f"{base_url}{user}")
        if response.status_code == 200:
            api_response.append(response.json())
        else:
            print("Error Occured")
        
    return api_response

#### Defining schema for json data

In [0]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("maidenName", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("username", StringType(), True),
    StructField("password", StringType(), True),
    StructField("birthDate", StringType(), True),
    StructField("image", StringType(), True),
    StructField("bloodGroup", StringType(), True),
    StructField("height", DoubleType(), True),
    StructField("weight", DoubleType(), True),
    StructField("eyeColor", StringType(), True),

    # hair struct
    StructField("hair", StructType([
        StructField("color", StringType(), True),
        StructField("type", StringType(), True)
    ]), True),

    StructField("ip", StringType(), True),

    # address struct
    StructField("address", StructType([
        StructField("address", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("stateCode", StringType(), True),
        StructField("postalCode", StringType(), True),

        StructField("coordinates", StructType([
            StructField("lat", DoubleType(), True),
            StructField("lng", DoubleType(), True)
        ]), True),

        StructField("country", StringType(), True)
    ]), True),

    StructField("macAddress", StringType(), True),
    StructField("university", StringType(), True),

    # bank struct
    StructField("bank", StructType([
        StructField("cardExpire", StringType(), True),
        StructField("cardNumber", StringType(), True),
        StructField("cardType", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("iban", StringType(), True)
    ]), True),

    # company struct
    StructField("company", StructType([
        StructField("department", StringType(), True),
        StructField("name", StringType(), True),
        StructField("title", StringType(), True),

        StructField("address", StructType([
            StructField("address", StringType(), True),
            StructField("city", StringType(), True),
            StructField("state", StringType(), True),
            StructField("stateCode", StringType(), True),
            StructField("postalCode", StringType(), True),

            StructField("coordinates", StructType([
                StructField("lat", DoubleType(), True),
                StructField("lng", DoubleType(), True)
            ]), True),

            StructField("country", StringType(), True)
        ]), True)
    ]), True),

    StructField("ein", StringType(), True),
    StructField("ssn", StringType(), True),
    StructField("userAgent", StringType(), True),

    # crypto struct
    StructField("crypto", StructType([
        StructField("coin", StringType(), True),
        StructField("wallet", StringType(), True),
        StructField("network", StringType(), True)
    ]), True),

    StructField("role", StringType(), True)
])

#### Creating dataframe from the json data

In [0]:
api_df = spark.createDataFrame(api_hit(), schema = schema)

In [0]:
api_df.limit(5).display()

id,firstName,lastName,maidenName,age,gender,email,phone,username,password,birthDate,image,bloodGroup,height,weight,eyeColor,hair,ip,address,macAddress,university,bank,company,ein,ssn,userAgent,crypto,role
1,Emily,Johnson,Smith,29,female,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,emilyspass,1996-5-30,https://dummyjson.com/icon/emilys/128,O-,193.24,63.16,Green,"List(Brown, Curly)",42.48.100.32,"List(626 Main Street, Phoenix, Mississippi, MS, 29112, List(-77.16213, -92.084824), United States)",47:fa:41:18:ec:eb,University of Wisconsin--Madison,"List(05/28, 3693233511855044, Diners Club International, GBP, GB74MH2UZLR9TRPHYNU8F8)","List(Engineering, Dooley, Kozey and Cronin, Sales Manager, List(263 Tenth Street, San Francisco, Wisconsin, WI, 37657, List(71.814525, -161.150263), United States))",977-175,900-590-289,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.93 Safari/537.36","List(Bitcoin, 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a, Ethereum (ERC20))",admin
2,Michael,Williams,,36,male,michael.williams@x.dummyjson.com,+49 258-627-6644,michaelw,michaelwpass,1989-8-10,https://dummyjson.com/icon/michaelw/128,B+,186.22,76.32,Red,"List(Green, Straight)",12.13.116.142,"List(385 Fifth Street, Houston, Alabama, AL, 38807, List(22.815468, 115.608581), United States)",79:15:78:99:60:aa,Ohio State University,"List(01/30, 3530633803003665, JCB, USD, DE26362283149158045865)","List(Support, Spinka - Dickinson, Support Specialist, List(395 Main Street, Los Angeles, New Hampshire, NH, 73442, List(79.098326, -119.624845), United States))",912-602,108-953-962,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/97.0.1072.76 Safari/537.36","List(Bitcoin, 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a, Ethereum (ERC20))",admin
3,Sophia,Brown,,43,female,sophia.brown@x.dummyjson.com,+81 210-652-2785,sophiab,sophiabpass,1982-11-6,https://dummyjson.com/icon/sophiab/128,O-,177.72,52.6,Hazel,"List(White, Wavy)",214.225.51.195,"List(1642 Ninth Street, Washington, Alabama, AL, 32822, List(45.289366, 46.832664), United States)",12:a3:d3:6f:5c:5b,Pepperdine University,"List(10/27, 6011212053392887, Discover, EUR, DE12191213468288004835)","List(Research and Development, Schiller - Zieme, Accountant, List(1896 Washington Street, Dallas, Nevada, NV, 88511, List(20.086743, -34.577107), United States))",963-113,638-461-822,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.45 Safari/537.36","List(Bitcoin, 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a, Ethereum (ERC20))",admin
4,James,Davis,,46,male,james.davis@x.dummyjson.com,+49 614-958-9364,jamesd,jamesdpass,1979-5-4,https://dummyjson.com/icon/jamesd/128,AB+,193.31,62.1,Amber,"List(Blonde, Straight)",101.118.131.66,"List(238 Jefferson Street, Seattle, Pennsylvania, PA, 68354, List(16.782513, -139.34723), United States)",10:7d:df:1f:97:58,University of Southern California,"List(07/30, 5303440212268149, Mastercard, CAD, DE01300746880579852937)","List(Support, Pagac and Sons, Research Analyst, List(1622 Lincoln Street, Fort Worth, Pennsylvania, PA, 27768, List(54.91193, -79.498328), United States))",904-810,116-951-314,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36","List(Bitcoin, 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a, Ethereum (ERC20))",admin
5,Emma,Miller,Johnson,31,female,emma.miller@x.dummyjson.com,+91 759-776-1614,emmaj,emmajpass,1994-6-13,https://dummyjson.com/icon/emmaj/128,AB-,192.8,63.62,Green,"List(White, Straight)",224.126.22.183,"List(607 Fourth Street, Jacksonville, Colorado, CO, 26593, List(0.505589, -157.43281), United States)",32:b9:7e:8d:f5:e8,Northeastern University,"List(07/30, 5237188057591130, Mastercard, NZD, DE19182355652037133559)","List(Human Resources, Graham - Gulgowski, Quality Assurance Engineer, List(1460 Sixth Street, San Antonio, Idaho, ID, 21965, List(44.346545,

#### Flattening the json data and selecting only the required columns

In [0]:
df1 = api_df.select(
    "id",
    "firstName",
    "lastName",
    "maidenName",
    "age",
    "gender",
    "username",
    "birthDate",
    "address.address",
    "address.city",
    "address.state",
    "address.coordinates.lat",
    "address.coordinates.lng",
    "address.country"
)

In [0]:
df1.limit(5).display()

id,firstName,lastName,maidenName,age,gender,username,birthDate,address,city,state,lat,lng,country
1,Emily,Johnson,Smith,29,female,emilys,1996-5-30,626 Main Street,Phoenix,Mississippi,-77.16213,-92.084824,United States
2,Michael,Williams,,36,male,michaelw,1989-8-10,385 Fifth Street,Houston,Alabama,22.815468,115.608581,United States
3,Sophia,Brown,,43,female,sophiab,1982-11-6,1642 Ninth Street,Washington,Alabama,45.289366,46.832664,United States
4,James,Davis,,46,male,jamesd,1979-5-4,238 Jefferson Street,Seattle,Pennsylvania,16.782513,-139.34723,United States
5,Emma,Miller,Johnson,31,female,emmaj,1994-6-13,607 Fourth Street,Jacksonville,Colorado,0.505589,-157.43281,United States


#### Joining dataframes with data from csv and api
- Enriching the csv dataframe
- df.join(df1, df.id == df1.id, "left") -> the joined table has two id columns
- df.join(df1, on = "id", how = "left") -> the joined table has single id column

In [0]:
new_df = df.join(df1, on = "id", how = "left")

In [0]:
new_df.limit(5).display()

id,transaction_date,purchase_amount,product_category,store_id,firstName,lastName,maidenName,age,gender,username,birthDate,address,city,state,lat,lng,country
1,2025-01-01,120.5,Electronics,S001,Emily,Johnson,Smith,29,female,emilys,1996-5-30,626 Main Street,Phoenix,Mississippi,-77.16213,-92.084824,United States
2,2025-01-02,45.0,Books,S002,Michael,Williams,,36,male,michaelw,1989-8-10,385 Fifth Street,Houston,Alabama,22.815468,115.608581,United States
3,2025-01-03,89.99,Clothing,S001,Sophia,Brown,,43,female,sophiab,1982-11-6,1642 Ninth Street,Washington,Alabama,45.289366,46.832664,United States
4,2025-01-04,230.1,Home,S003,James,Davis,,46,male,jamesd,1979-5-4,238 Jefferson Street,Seattle,Pennsylvania,16.782513,-139.34723,United States
5,2025-01-05,15.75,Books,S002,Emma,Miller,Johnson,31,female,emmaj,1994-6-13,607 Fourth Street,Jacksonville,Colorado,0.505589,-157.43281,United States


##### Replacing the blankspaces with n/a

In [0]:
new_df = new_df.withColumn("maidenName", F.when(col("maidenName") == "", "n/a").otherwise(col("maidenName")))

In [0]:
new_df.limit(5).display()

id,transaction_date,purchase_amount,product_category,store_id,firstName,lastName,maidenName,age,gender,username,birthDate,address,city,state,lat,lng,country
1,2025-01-01,120.5,Electronics,S001,Emily,Johnson,Smith,29,female,emilys,1996-5-30,626 Main Street,Phoenix,Mississippi,-77.16213,-92.084824,United States
2,2025-01-02,45.0,Books,S002,Michael,Williams,n/a,36,male,michaelw,1989-8-10,385 Fifth Street,Houston,Alabama,22.815468,115.608581,United States
3,2025-01-03,89.99,Clothing,S001,Sophia,Brown,n/a,43,female,sophiab,1982-11-6,1642 Ninth Street,Washington,Alabama,45.289366,46.832664,United States
4,2025-01-04,230.1,Home,S003,James,Davis,n/a,46,male,jamesd,1979-5-4,238 Jefferson Street,Seattle,Pennsylvania,16.782513,-139.34723,United States
5,2025-01-05,15.75,Books,S002,Emma,Miller,Johnson,31,female,emmaj,1994-6-13,607 Fourth Street,Jacksonville,Colorado,0.505589,-157.43281,United States


#### Writing the enriched dataframe onto a table

In [0]:
new_df.write.mode("overwrite").saveAsTable("workspace.streaming.api_table")